# Phase 4: Full Prescription Inference Demo

This notebook runs the complete inference pipeline from full prescription images to OCR and lexicon-matched medication predictions.

Input: full prescription image(s).

Output: processed pages, region crops, line crops, word crops, OCR predictions, medicine lexicon matching, dosage/frequency extraction, overview images, CSV, and JSON.

## Step 1: Mount Drive, Pull Latest Code, and Enter Repo

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nbl-ahmd/project.git'
DRIVE_BASE = Path('/content/drive/MyDrive/phase4_project')
REPO_DIR = DRIVE_BASE / 'repo'
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    if not (REPO_DIR / 'pipeline').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
elif IN_KAGGLE:
    REPO_DIR = Path('/kaggle/working/project')
    if not (REPO_DIR / 'pipeline').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    REPO_DIR = Path.cwd()

os.chdir(REPO_DIR)
print('Repository:', Path.cwd())
print('IN_COLAB:', IN_COLAB, 'IN_KAGGLE:', IN_KAGGLE)
print('Has pipeline:', (Path.cwd() / 'pipeline').exists())

## Step 2: Install Dependencies

Use a GPU runtime for TrOCR inference if available. CPU works for a few words but will be slow.

In [ ]:
!python3 -m pip install -q -r pipeline/requirements.txt
!python3 -m pip install -q -r pipeline/requirements-layout.txt
!python3 -m pip install -q -r pipeline/requirements-ocr.txt

## Step 3: Configure Input Images and Models

Put full prescription images in `data/full_inference_input/`, or change `INPUT_PATH` to one image file.

For final results, set `TROCR_MODEL` to your fine-tuned `best_model` folder. If no fine-tuned model is found, the notebook falls back to base TrOCR for testing only.

In [ ]:
from pathlib import Path

INPUT_PATH = Path('data/full_inference_input')
INPUT_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path('data/full_pipeline_inference_demo')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional region detector. If missing, pipeline uses a heuristic region proposal.
REGION_MODEL = Path('models/region_yolo_best.pt')

# Fine-tuned TrOCR model candidates.
MODEL_CANDIDATES = [
    Path('/content/drive/MyDrive/phase4_project/trocr_work/best_model'),
    Path('/kaggle/working/trocr_work/best_model'),
    Path('trocr_work/best_model'),
    Path('models/trocr_best_model'),
]
existing_models = [p for p in MODEL_CANDIDATES if p.exists()]
TROCR_MODEL = str(existing_models[0]) if existing_models else 'microsoft/trocr-base-handwritten'

LEXICON_PATH = Path('pipeline/config/drug_lexicon.txt')

# Keep this lower for quick demos; use 2200 for final high-quality runs.
MAX_SIDE = 1600
NUM_BEAMS = 1
MAX_TARGET_LEN = 48

print('Input path:', INPUT_PATH.resolve())
print('Input image count:', len([p for p in INPUT_PATH.glob('*') if p.is_file()]))
print('Output dir:', OUTPUT_DIR.resolve())
print('Region model:', REGION_MODEL.resolve(), 'exists=', REGION_MODEL.exists())
print('TrOCR model:', TROCR_MODEL)
print('Lexicon:', LEXICON_PATH.resolve(), 'exists=', LEXICON_PATH.exists())

if TROCR_MODEL == 'microsoft/trocr-base-handwritten':
    print('WARNING: fine-tuned TrOCR model not found. Base TrOCR is only for testing, not final results.')

## Step 4: Upload Full Prescription Images, If Needed

Run this cell in Colab if `data/full_inference_input/` is empty.

In [ ]:
if IN_COLAB and len([p for p in INPUT_PATH.glob('*') if p.is_file()]) == 0:
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        out = INPUT_PATH / name
        out.write_bytes(content)
    print('Uploaded files:', list(uploaded.keys()))
else:
    print('Skipping upload. Existing files:', [p.name for p in INPUT_PATH.glob('*') if p.is_file()][:10])

## Step 5: Preview Input Images

In [ ]:
from IPython.display import display, Image as IPImage

image_files = [p for p in sorted(INPUT_PATH.glob('*')) if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff']]
if not image_files:
    raise FileNotFoundError(f'No input images found in {INPUT_PATH}. Upload images or update INPUT_PATH.')

for p in image_files[:5]:
    print(p)
    display(IPImage(filename=str(p), width=700))

## Step 6: Run Full Pipeline Inference

This runs:

full image -> preprocessing -> region crop -> line segmentation -> word segmentation -> TrOCR OCR -> lexicon/dosage/frequency validation.

In [ ]:
import subprocess

cmd = [
    'python3', 'pipeline/scripts/run_end_to_end.py',
    '--input', str(INPUT_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--ocr-backend', 'trocr',
    '--ocr-unit', 'word',
    '--trocr-model', str(TROCR_MODEL),
    '--max-target-len', str(MAX_TARGET_LEN),
    '--num-beams', str(NUM_BEAMS),
    '--line-padding', '6',
    '--max-side', str(MAX_SIDE),
    '--lexicon', str(LEXICON_PATH),
]

if REGION_MODEL.exists():
    cmd += ['--yolo-model', str(REGION_MODEL), '--target-class', '0']
else:
    print('Region model missing. Using heuristic region proposal fallback.')

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## Step 7: Load Output Tables

In [ ]:
import pandas as pd

page_manifest = OUTPUT_DIR / 'page_manifest.csv'
region_manifest = OUTPUT_DIR / 'region_manifest.csv'
line_manifest = OUTPUT_DIR / 'line_manifest.csv'
word_manifest = OUTPUT_DIR / 'word_manifest.csv'
review_csv = OUTPUT_DIR / 'segmentation_review.csv'
pred_csv = OUTPUT_DIR / 'predictions.csv'

pages = pd.read_csv(page_manifest) if page_manifest.exists() else pd.DataFrame()
regions = pd.read_csv(region_manifest) if region_manifest.exists() else pd.DataFrame()
lines = pd.read_csv(line_manifest) if line_manifest.exists() else pd.DataFrame()
words = pd.read_csv(word_manifest) if word_manifest.exists() else pd.DataFrame()
review = pd.read_csv(review_csv) if review_csv.exists() else pd.DataFrame()
pred = pd.read_csv(pred_csv) if pred_csv.exists() else pd.DataFrame()

print('Pages:', len(pages))
print('Regions:', len(regions))
print('Lines:', len(lines))
print('Words:', len(words))
print('Predictions:', len(pred))
display(pred.head(30))

## Step 8: Show Segmentation Overview Images

In [ ]:
overview_dir = OUTPUT_DIR / 'segmentation_overview'
print('Overview folder:', overview_dir)

print('Line overviews')
for p in sorted(overview_dir.glob('*line_overview.png'))[:5]:
    print(p)
    display(IPImage(filename=str(p), width=950))

print('Word overviews')
for p in sorted(overview_dir.glob('*word_overview.png'))[:8]:
    print(p)
    display(IPImage(filename=str(p), width=950))

## Step 9: Show OCR Result Table and Validated Medicines

In [ ]:
if pred.empty:
    raise RuntimeError('No predictions found. Check the inference command output above.')

cols = ['page_id', 'region_id', 'line_id', 'word_id', 'image_path', 'ocr_text', 'medicine_name', 'medicine_match_score', 'dosage', 'frequency', 'validation_status']
display(pred[[c for c in cols if c in pred.columns]].head(60))

matched = pred[pred['validation_status'].astype(str) == 'matched'].copy() if 'validation_status' in pred.columns else pd.DataFrame()
print('Matched medicine rows:', len(matched))
display(matched[[c for c in cols if c in matched.columns]].head(40))

## Step 10: Create Demo OCR Grid Image

In [ ]:
import math
import cv2
import numpy as np

preview_dir = OUTPUT_DIR / 'demo_preview'
preview_dir.mkdir(parents=True, exist_ok=True)

def make_card(row):
    image_path = Path(str(row['image_path']))
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    target_w = 360
    scale = target_w / max(1, w)
    resized = cv2.resize(img, (target_w, max(36, int(h * scale))), interpolation=cv2.INTER_AREA)
    canvas = np.full((resized.shape[0] + 86, target_w, 3), 255, dtype=np.uint8)
    canvas[:resized.shape[0], :, :] = resized
    line1 = f"OCR: {row.get('ocr_text', '')}"[:38]
    line2 = f"Match: {row.get('medicine_name', '')} ({row.get('medicine_match_score', '')})"[:42]
    line3 = f"{row.get('validation_status', '')}"[:38]
    y0 = resized.shape[0]
    cv2.putText(canvas, line1, (8, y0 + 24), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (20, 20, 20), 1, cv2.LINE_AA)
    cv2.putText(canvas, line2, (8, y0 + 50), cv2.FONT_HERSHEY_SIMPLEX, 0.43, (0, 90, 180), 1, cv2.LINE_AA)
    cv2.putText(canvas, line3, (8, y0 + 74), cv2.FONT_HERSHEY_SIMPLEX, 0.43, (0, 120, 0), 1, cv2.LINE_AA)
    return canvas

cards = []
for _, row in pred.head(30).iterrows():
    card = make_card(row)
    if card is not None:
        cards.append(card)

if cards:
    cols = 3
    card_h = max(c.shape[0] for c in cards)
    card_w = cards[0].shape[1]
    rows_n = math.ceil(len(cards) / cols)
    grid = np.full((rows_n * card_h, cols * card_w, 3), 245, dtype=np.uint8)
    for idx, card in enumerate(cards):
        y = (idx // cols) * card_h
        x = (idx % cols) * card_w
        grid[y:y+card.shape[0], x:x+card.shape[1]] = card
    preview_path = preview_dir / 'full_pipeline_ocr_grid.png'
    cv2.imwrite(str(preview_path), grid)
    print('Saved:', preview_path)
    display(IPImage(filename=str(preview_path), width=1000))
else:
    print('No cards created.')

## Step 11: Final Output Summary

In [ ]:
print('Final output folder:', OUTPUT_DIR)
print('Predictions CSV:', OUTPUT_DIR / 'predictions.csv')
print('Predictions JSON:', OUTPUT_DIR / 'predictions.json')
print('Segmentation overview images:', OUTPUT_DIR / 'segmentation_overview')
print('Demo OCR grid:', OUTPUT_DIR / 'demo_preview' / 'full_pipeline_ocr_grid.png')

if 'validation_status' in pred.columns:
    display(pred['validation_status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count'))
if not review.empty:
    display(review.head(30))